# Spatial Clustering: NYC Uber Pickups

Dataset: FiveThirtyEight Uber pickups, April 2014, loaded with `sc4020.datasets.load_uber_pickups()`:
15,000 pickups inside NYC, projected to km. There is no ground truth, so only internal metrics
(silhouette, Davies-Bouldin) are reported.

Goal: discover urban transit hotspots and isolate outer-borough pickups as noise.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sc4020 import generate_figures as gf
from sc4020 import methods, tuning
from sc4020.datasets import load_uber_pickups
from sc4020.paths import results_dir

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
METRIC_COLS = [
    "method", "fit_seconds", "n_clusters", "noise_fraction", "largest_cluster_share",
    "silhouette", "davies_bouldin",
]

## Load and filter

Pickups outside the NYC bounding box are dropped, then 15,000 points are sampled for speed
and clean plots.

In [ ]:
ds = load_uber_pickups(n_samples=15000, random_state=42)
print(f"April 2014 file: {ds.n_rows_raw:,} rows -> {ds.n_rows_in_nyc:,} inside NYC -> sampled X={ds.data.shape} (km)")

## Fit clustering models

All four methods are set by the shared rule (`tuning.select_all`) on the km projection: K-Means
k by silhouette, GMM k and covariance by BIC, DBSCAN and HDBSCAN by the best silhouette with at
least 2 clusters and at most 50% noise. Because the data is in km, DBSCAN's eps is a distance in km.

In [ ]:
# All four methods by the shared rule on the km projection; silhouette estimated on 5,000 points.
selections = tuning.select_all(ds.data, sample_size=5000)
params = {m: s.params for m, s in selections.items()}
results = methods.run_models(methods.build_models(params), ds.data, params=params)
for name, r in results.items():
    print(f"{name:10s} params={r.params}  fit_seconds={r.fit_seconds:.3f}")

## Metrics

Silhouette is computed on a 3,000-point subsample to keep it fast.

In [ ]:
results_df = gf.build_results_table("uber_pickups", ds.data, results, ds.target, silhouette_sample=3000)
display(results_df[METRIC_COLS].round(4))
results_df.to_csv(results_dir("datasets") / "uber_pickups_results.csv", index=False)

## Spatial visualization

In [ ]:
gf.plot_spatial_grid(
    ds.coords_deg, results,
    title="Hotspot Cluster Analysis on NYC Uber Pickups",
    reference_title="Actual Pickup Density (15k points)",
)
plt.show()

## Discussion

* **DBSCAN geometric alignment:** traces the elongated density corridors of Manhattan and
  isolates major hubs (JFK in the southeast, LaGuardia in the north) as distinct components.
* **Noise filtering:** dispersed pickups across Staten Island, deep Brooklyn and Queens fall
  into the noise label (`-1`) instead of skewing centroids.
* **Partitioning models (K-Means & GMM):** they must assign every point to a cluster, so
  isolated suburban trips get arbitrary radial boundaries and the continuous Manhattan
  corridor is cut into rigid convex cells.